In [1]:
import dotenv
dotenv.load_dotenv(override=True)

import textmancy
import pandas as pd

# Benchmark Annotations
First, we benchmark the annotating capacity of textmancy. We will use the [ecommerce dataset](https://www.kaggle.com/datasets/saurabhshahane/ecommerce-text-classification) from Kaggle.
This dataset contains ecommerce product descriptions, categorized into 4 categories: "Electronics", "Household", "Books" and "Clothing & Accessories". We will use these categories to annotate the descriptions.



In [2]:
import csv
import pydantic
from textmancy.components import Annotator

# Read file as tuples
reader = csv.reader(open('sample_data/ecommerceDataset.csv', 'r', encoding='utf-8'))
ecommerce_dataset = []
for row in reader:
    ecommerce_dataset.append(row)

# Create targets
class ProductCategory(pydantic.BaseModel):
    """
    A category of ecommerce goods product. Represents high-level classification of products.

    """
    name: str
    description: str

product_categories = [
    ProductCategory(name='Electronics', description='Electronic devices and accessories.'),
    ProductCategory(name='Household', description='Household goods and accessories.'),
    ProductCategory(name='Books', description='Books and reading materials.'),
    ProductCategory(name='Clothing & Accessories', description='Beauty products and accessories.'),
]

annotator = Annotator(
    targets=product_categories,
)

## Single Annotation

We start with a direct task, annotating each product description with a single category. We will use the `annotate` method of the `TextAnnotator` class to annotate

We sample from the dataset and annotate

In [6]:

from random import sample, seed

seed(42)
test_set = sample(ecommerce_dataset, 100)

annotations = [annotator.annotate(text=row[1], chunk_size=8000) for row in test_set]


,text,predicted,actual,accuracy
count,100,95,100,100
unique,100,4,4,2
top,Generic 600Mbps USB Wifi Dongle 600Mbps Wirele...,Electronics,Household,True
freq,1,36,46,65


In [9]:
result = pd.DataFrame(
    {
        "text": [r[1] for r in test_set],
        "predicted": [a[0].name if a else None for a in annotations],
        "actual": [r[0] for r in test_set],
    }
)
result["accuracy"] = result["predicted"] == result["actual"]
print(result["accuracy"].mean())

print(result.query("accuracy == False").head(10))

0.65
                                                 text  \
5   USHA Plastic Fiber Tower Fans (White and Black...   
6   Voltas 1 Ton 3 Star Split AC (Copper, 123 CZA,...   
7   Britannia Little Hearts , 37g The iconic gold ...   
12  Alan Jones Clothing Men's Cotton T-Shirt (Stc-...   
14  SOJOS Fashion Womens Sunglasses Oversized Squa...   
19  Rodak Round Furniture Brush, 35 Mm Inner Diame...   
21  Kuchipoo Premium Quality Boys Lower Kids Track...   
22  Claura Grey Printed Slim Fit Ankle Length Spor...   
24  Ziya Women's Pure Cotton Camisole for Salwar S...   
25  Zollyss Plastic Mini Electric 7 Egg Poacher St...   

                   predicted                  actual  accuracy  
5                Electronics               Household     False  
6                Electronics               Household     False  
7                       None               Household     False  
12  Clothing and Accessories  Clothing & Accessories     False  
14               Electronics  Clothing & A